In [1]:
# ============================================
# Outlier Detection using K-Means + NLP
# Dataset: outlier_info.csv
# ============================================

# 1. Import libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics import silhouette_score


# ============================================
# 2. Load Dataset
# ============================================

df = pd.read_csv("outlier_info.csv")

print(df.head())
print(df.info())



# ============================================
# 3. Check Missing Values
# ============================================

print(df.isnull().sum())

df = df.dropna()



# ============================================
# 4. Separate Text and Numeric Columns
# ============================================

# Numeric columns
numeric_cols = df.select_dtypes(
    include=['int64','float64']
).columns

# Text columns
text_cols = df.select_dtypes(
    include=['object']
).columns


print("Numeric columns:", numeric_cols)
print("Text columns:", text_cols)



# ============================================
# 5. NLP Feature Extraction
# ============================================

# Combine all text columns
if len(text_cols) > 0:

    df["combined_text"] = df[text_cols].astype(str).agg(
        ' '.join, axis=1
    )

    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=500
    )

    text_features = vectorizer.fit_transform(
        df["combined_text"]
    )

    text_features = text_features.toarray()

else:
    text_features = np.empty(
        (len(df),0)
    )


# ============================================
# 6. Scale Numeric Features
# ============================================

if len(numeric_cols) > 0:

    scaler = StandardScaler()

    numeric_features = scaler.fit_transform(
        df[numeric_cols]
    )

else:
    numeric_features = np.empty(
        (len(df),0)
    )



# ============================================
# 7. Combine NLP + Numeric Features
# ============================================

X = np.hstack(
    [
        numeric_features,
        text_features
    ]
)

print("Final feature shape:", X.shape)



# ============================================
# 8. Find Best K using Silhouette Score
# ============================================

scores = []

for k in range(2,10):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X)

    score = silhouette_score(
        X,
        labels
    )

    scores.append(score)


plt.plot(
    range(2,10),
    scores,
    marker='o'
)

plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")
plt.title("Finding Optimal K")
plt.show()



# ============================================
# 9. Apply K-Means
# ============================================

optimal_k = np.argmax(scores)+2

kmeans = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X)


print(df["cluster"].value_counts())



# ============================================
# 10. Detect Outliers
# ============================================

# Distance from cluster center

distances = kmeans.transform(X)

df["distance_to_center"] = np.min(
    distances,
    axis=1
)


# Define outliers using percentile threshold

threshold = df["distance_to_center"].quantile(
    0.95
)


df["outlier"] = np.where(
    df["distance_to_center"] > threshold,
    "Yes",
    "No"
)


# Show detected outliers

outliers = df[
    df["outlier"]=="Yes"
]

print(
    "Number of outliers:",
    len(outliers)
)

outliers.head()



# ============================================
# 11. Visualise Clusters using PCA
# ============================================

pca = PCA(
    n_components=2
)

X_pca = pca.fit_transform(X)


plt.figure(figsize=(8,6))

plt.scatter(
    X_pca[:,0],
    X_pca[:,1],
    c=df["cluster"]
)

plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("K-Means Clusters")

plt.show()



# ============================================
# 12. Save Results
# ============================================

df.to_csv(
    "outlier_results.csv",
    index=False
)

print(
    "Saved: outlier_results.csv"
)

             Unnamed: 0  fixed acidity  volatile acidity  citric acid  \
0    Number of Outliers         106.00            133.00       223.00   
1   Outliers Percentage           2.68              3.36         5.63   
2          75% Quantile           7.30              0.33         0.39   
3          25% Quantile           6.30              0.21         0.27   
4  Inter Quantile Range           1.00              0.12         0.12   

   residual sugar  chlorides  free sulfur dioxide  total sulfur dioxide  \
0            16.0    178.000                44.00                 14.00   
1             0.4      4.490                 1.11                  0.35   
2             8.9      0.050                45.00                166.00   
3             1.6      0.035                23.00                106.00   
4             7.3      0.015                22.00                 60.00   

   density     pH  sulphates  alcohol  quality  
0  6.00000  46.00      96.00      0.0   156.00  
1  0.15000  

/var/folders/88/w4w1n8l12kd_z42_6mrnndmw0000gn/T/ipykernel_73785/3120181348.py:51: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(


ValueError: Number of labels is 7. Valid values are 2 to n_samples - 1 (inclusive)